# Lab : IBM-Models for Machine Translation

## Objectives:

Implement the IBM-1 and IBM-2 alignment models for Machine Translation
1. Look at the translation data: understand what we have access to, collect some statistics 
2. Implement a IBM1 model
    - From scratch: we use a *translation table* ```t``` with uniform alignements.
    - We use the EM algorithm to update ```t``` using parallel data
        - The latent variable (*unobserved*) is the alignment 
3. Implement a IBM2 model
    - From scratch: we use a *translation table* ```t``` and an alignement table ```a```.
    - We use the EM algorithm to update ```t``` and ```a``` using parallel data
4. Compare with the NLTK implementations
5. Implement a **very naive** translation function

### Necessary dependencies

We will need the Natural Language Toolkit : http://www.nltk.org/install.html for preprocessing and to compare our implementation with existing IBM1 and 2 models

In [15]:
from io import open
import re
import string
import math
import numpy as np

# NLTK
import nltk
nltk.download('punkt')
from nltk import word_tokenize

# Nice printing
from pprint import pprint

# Format for translation table
from collections import defaultdict

[nltk_data] Downloading package punkt to /home/matthieu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


We're going to work with data from the **tatoeba** website. This website proposes human-made translations for many (relatively) simple sentences, with sometimes several possible translations for one sentence. 
Pre-processed versions of the *tatoeba dataset* can be found on this [website](https://www.manythings.org/anki/). 

In [2]:
# Read the file and split into lines
parallel = open('fra.txt', encoding='utf-8').\
        read().strip().split('\n')

In [3]:
# Data looks like this
pprint(parallel[0:5])

['Go.\tVa !\tCC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & '
 '#1158250 (Wittydev)',
 'Go.\tMarche.\tCC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & '
 '#8090732 (Micsmithel)',
 'Go.\tEn route !\tCC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & '
 '#8267435 (felix63)',
 'Go.\tBouge !\tCC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & '
 '#9022935 (Micsmithel)',
 'Hi.\tSalut !\tCC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & '
 '#509819 (Aiji)']


### I - Exploring data

Explore the *training data*: 
- Apply **normalization** and **pre-processing**
- Count the number of sentences, of words for each language, visualize their frequency
- Display an histogram of sentence lengths to have an idea of how the sequences are distributed 
- Check if sentences in both languages are of different lengths
<div class='alert alert-block alert-info'>
            Code:</div>

In [4]:
# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = s.lower().strip()
    regex = re.compile('[%s]' % re.escape(string.punctuation))
    s = regex.sub(" ", s)
    # s = re.sub(r'[^A-Za-zÀ-ÖØ-öø-ÿ/]+', r" ", s)
    return s.strip()

In [5]:
# Split every line into pairs and normalize
pairs = [[normalizeString(s) for s in l.split('\t')[:2]] for l in parallel]

In [6]:
# Look at the dataset size
print(len(pairs))

232736


In [7]:
# Transform the data to get the parallel sentences tokenized
def preprocess(sent):
    return [w.lower() for w in word_tokenize(sent)]

bitext = [
    (["NULL"] + preprocess(fr), preprocess(en))
    for fr, en in pairs
]    

In [8]:
pprint(bitext[:5])

[(['NULL', 'go'], ['va']),
 (['NULL', 'go'], ['marche']),
 (['NULL', 'go'], ['en', 'route']),
 (['NULL', 'go'], ['bouge']),
 (['NULL', 'hi'], ['salut'])]


### II - Implementing a IBM-1 model

We will now implement a ```train_ibm1(parallel_data, num_iters):``` function.
It will require:
- A ```initialize_t(parallel_data)``` function, which initializes the translation table $t$
- A ```compute_log_likelihood_ibm1(parallel_data, t)``` function, which computes the log-likelihood given by the model to the parallel data

The function ```train_ibm1``` will perform **Expectation-Maximization**, as we did in class, to optimize the parameters in $t$.
The algorithm is detailled as follows:

#### IBM-1: Training algorithm

- **Input:** A training corpus $(f^k, e^k)$ for $k = 1,...,n$, where $f^k = (f_1^k,...,f_{l_f^k}^k)$ and $e^k = (e_1^k,...,e_{l_e^k}^k)$
- **Initialization:** Initialize $t(f |e)$ to uniform probabilities $\forall f, e$ words in the source and target languages
- For $s = 1...S$ (*Iterations of the EM algorithm*)
    - Set all counts $c(...) = 0$
    - **E-step**
        - For $k = 1...n$ (*Data samples*)
            - For $i = 1...l_f^k$ (*Length of source sentence*)
                - For $j = 0...l_e^k$ (*Length of target sentence*)
                    - $p(f_i^k |e_j^k) = \frac{t(f_i^k |e_j^k)}{\sum_{j = 0}^{l_e^k} t(f_i^k |e_j^k)}$ (*Alignment probability: it is the word-to-word translation probability, positional alignment is assumed uniform*)
                    - $c(e_j^k, f_i^k) = c(e_j^k, f_i^k) +  p(f_i^k |e_j^k)$
                    - $c(e_j^k) = c(e_j^k) +  p(f_i^k |e_j^k)$
    - **M-step**
        - $\forall f, e$, update $t(f |e) = \frac{c(e, f)}{c(e)}$ (*Empirical probability, as always - can be proven to maximise the **complete** likelihood if knowling the alignment*)
- **Output**: parameters $t(f |e)$


Notes on implementation:
- Rather than looking for an efficient implementation doing vector operations, we will go for the naïve way and work with loops, using ```defautdict``` to keep track of the translation table $t$ and counts $c$ - it facilitates dealing with vocabularies
- This function will be **slow**: limit the size of the training data (do not take the full datase !) and the number of iterations

<div class='alert alert-block alert-info'>
            Code:</div>

In [39]:
# Initialize a translation table on the data
def initialize_t(bitext):
    # For ease of use (rather than efficiency) use defaultdict
    t = defaultdict(lambda: defaultdict(float))
    vocab_e = set()
    vocab_f = set()
    # Fill both vocabularies 
    for e_sent, f_sent in bitext:
        vocab_e.update(e_sent)
        vocab_f.update(f_sent)
    # Initialize with uniform probabilities
    initial_prob = 1.0 / len(vocab_f)
    for e in vocab_e:
        for f in vocab_f:
            t[e][f] = initial_prob
    return t

In [10]:
# Create a function to keep track of likelihood during EM-based learning
def compute_log_likelihood_ibm1(bitext, t):
    total_ll = 0.0
    # Loop over data
    for e_sent, f_sent in bitext:
        for f in f_sent:
            # Sum over all possible alignments
            prob_f = 0.0
            for e in e_sent:
                prob_f += t[e][f]
            total_ll += math.log(prob_f)
    return total_ll

In [13]:
# Apply a very simple EM model 
def train_ibm1(bitext, num_iters=10):
    # Initialize the table on data
    t = initialize_t(bitext)
    # Keep track of log-likelihood during training
    ll_history = []

    # EM iterations
    for iteration in range(num_iters):
        # We will keep track of the accumulation of alignements in there
        count = defaultdict(lambda: defaultdict(float))
        # We will keep track of the sum of probabilities of alignement with each target word in there
        total_e = defaultdict(float)

        # E-step: compute alignment probabilities for each sentence, cumulate all alignments in "count"
        for e_sent, f_sent in bitext:
            # For each origin word
            for f in f_sent:
                # Sum of current probabilities over all possible target positions
                Z = sum(t[e][f] for e in e_sent)
                # For each target position, compute the probability
                for e in e_sent:
                    c = t[e][f] / Z
                    # Cumulate our accumulation variables
                    count[e][f] += c
                    total_e[e] += c

        # M-step: update translation table "t" thanks to normalized alignments from "counts"
        for e in t:
            # Over all origin words "f" aligned with target words "e"
            for f in t[e]:
                if total_e[e] > 0:
                    t[e][f] = count[e][f] / total_e[e]

        # Compute log-likelihood
        ll = compute_log_likelihood_ibm1(bitext, t)
        ll_history.append(ll)
        print(f"IBM1 Iter {iteration+1}: Log-Likelihood = {ll:.2f}")                

    return t, ll_history

In [16]:
# Train the model on a subset of the data
t, ll_history = train_ibm1(bitext[:50000], num_iters=10)

Iteration 1
IBM1 Iter 1: Log-Likelihood = -456553.50
Iteration 2
IBM1 Iter 2: Log-Likelihood = -336238.77
Iteration 3
IBM1 Iter 3: Log-Likelihood = -310000.62
Iteration 4
IBM1 Iter 4: Log-Likelihood = -302167.62
Iteration 5
IBM1 Iter 5: Log-Likelihood = -299046.19
Iteration 6
IBM1 Iter 6: Log-Likelihood = -297560.79
Iteration 7
IBM1 Iter 7: Log-Likelihood = -296757.95
Iteration 8
IBM1 Iter 8: Log-Likelihood = -296279.38
Iteration 9
IBM1 Iter 9: Log-Likelihood = -295971.22
Iteration 10
IBM1 Iter 10: Log-Likelihood = -295760.59


In [17]:
# Check the values of probability tables
print(t['i']['je'])
print(t['you']['tu'])
print(t['you']['vous'])

0.5577261454618442
0.3713493257053642
0.47950637700087645


### III - Implementing a IBM-2 model

We will now implement a ```train_ibm2(parallel_data, num_iters, initial_t):``` function.
It will require, additionaly:
- A ```initialize_a(parallel_data)``` function, which initializes the alignment table $a$
- A modified ```compute_log_likelihood_ibm2(parallel_data, t, a)``` function, which computes the log-likelihood given by the model to the parallel data *using the alignment*

The function ```train_ibm2``` will also perform **Expectation-Maximization**, to optimize the parameters in $t$ **and** $a$.
The algorithm is detailled as follows:

#### IBM-2 Algorithm

**Input:** A training corpus $(f^k, e^k)$ for $k = 1,...,n$, where $f^k = (f_1^k,...,f_{l_f^k}^k)$ and $e^k = (e_1^k,...,e_{l_e^k}^k)$

**Initialization:** 
- Initialize $t(f |e)$ to uniform probabilities $\forall f, e$ 
- Initialize $a(i,j,l_f,l_e)$ to uniform probabilities $\frac{1}{l_e}$ for *every configuration observed in training data*

**Algorithm:**
- For $s = 1...S$ (*Iterations of the EM algorithm*)
    - Set all counts $c(...) = 0$
    - Set all *alignment* counts $c_a(...) = 0$
    - **E-step**
        - For $k = 1...n$ (*Data samples*)
            - For $i = 1...l_f^k$ (*Length of source sentence*)
                - For $j = 0...l_e^k$ (*Length of target sentence*)
                    - $\delta(k, i, j) = \frac{a(i,j,l_f^k,l_e^k) \times t(f_i^k |e_j^k)}{\sum_{j' = 0}^{l_e^k} a(i,j',l_f^k,l_e^k) \times \times t(f_i^k |e_{j'}^k)}$ (*Word-to-word translation probability multiplied by the positional alignment probability*)
                    - $c(e_j^k, f_i^k) = c(e_j^k, f_i^k) +  \delta(k, i, j)$
                    - $c(e_j^k) = c(e_j^k) +  \delta(k, i, j)$
                    - $c_a(i,j,l_f^k,l_e^k) = c_a(i,j,l_f^k,l_e^k) +  \delta(k, i, j)$
                    - $c_a(i,l_f^k,l_e^k) = c_a(i,l_f^k,l_e^k) +  \delta(k, i, j)$
    - **M-step**
        - $\forall f, e$, update $t(f |e) = \frac{c(e, f)}{c(e)}$
        - $\forall l_f, l_e$
            -  $\forall i, j$ update $a(i,j,l_f,l_e) = \frac{c_a(i,j,l_f,l_e)}{c_a(i,l_f,l_e)}$
- **Output**: parameters $t(f |e)$,  $a(i,j,l_f,l_e)$

<div class='alert alert-block alert-info'>
            Code:</div>

In [40]:
# Initialize the alignment table
def initialize_a(bitext):
    # Again, we'll use dictionnaries to make it easier - but very inefficient
    a = defaultdict(float)
    # For each parallel sentence, 
    for e_sent, f_sent in bitext:
        l_e = len(e_sent)
        l_f = len(f_sent)
        # Just initialize every existing "source length * target length" to uniform probabilities
        for j in range(l_f):
            for i in range(l_e):
                a[(i, j, l_e, l_f)] = 1.0 / l_e
    return a

In [41]:
# Compute the log likelihood for an IBM2 model: using alignment
def compute_log_likelihood_ibm2(bitext, t, a):
    total_ll = 0.0
    for e_sent, f_sent in bitext:
        l_e = len(e_sent)
        l_f = len(f_sent)
        # We use enumerate to keep track of positions
        for j, f in enumerate(f_sent):
            prob = 0.0
            # What changes: we multiply the probability of word translation by the probability of alignement for those positions 
            for i, e in enumerate(e_sent):
                prob += t[e][f] * a[(i, j, l_e, l_f)]
            total_ll += math.log(prob)
    return total_ll

In [42]:
def train_ibm2(bitext, num_iters=10, initial_t = None):
    # We can get an IBM1 model to initialize IBM2
    if initial_t is not None:
        t = initial_t
    else:
        t = initialize_t(bitext)
    # Initialize the alignemnts
    a = initialize_a(bitext)
    # Keep track of log-likelihood during training
    ll_history = []

    # EM iterations
    for iteration in range(num_iters):
        count_t = defaultdict(lambda: defaultdict(float))
        total_e = defaultdict(float)
        # Similarly, we accumulate alignement probabilities in there during the E-step
        count_a = defaultdict(float)
        # And we will use the sum for each receiving positions to normalize them during the M-step
        # Accumulate in there
        total_a = defaultdict(float)

        # E-step
        for e_sent, f_sent in bitext:
            l_e = len(e_sent)
            l_f = len(f_sent)
            # Looping over the data
            for j, f in enumerate(f_sent):
                # This time we need to loop to obtain the sum of probabilities of translation between "f" and every "e"
                Z = 0.0
                for i, e in enumerate(e_sent):
                    Z += t[e][f] * a[(i, j, l_e, l_f)]
                # And we can compute alignment probabilities
                for i, e in enumerate(e_sent):
                    delta = (t[e][f] * a[(i, j, l_e, l_f)]) / Z
                    # Update translation counts
                    count_t[e][f] += delta
                    total_e[e] += delta
                    # Update alignment counts
                    count_a[(i, j, l_e, l_f)] += delta
                    total_a[(j, l_e, l_f)] += delta

        # M-step
        # Update translation table "t"
        for e in count_t:
            for f in count_t[e]:
                t[e][f] = count_t[e][f] / total_e[e]
        # Update alignment table "a"
        for (i, j, l_e, l_f) in count_a:
            a[(i, j, l_e, l_f)] = (count_a[(i, j, l_e, l_f)] / total_a[(j, l_e, l_f)])

        # Compute log-likelihood
        ll = compute_log_likelihood_ibm2(bitext, t, a)
        ll_history.append(ll)
        print(f"IBM2 Iter {iteration+1}: Log-Likelihood = {ll:.2f}")
    return t, a, ll_history

In [43]:
# Train the model on a subset of the data
t2, a2, ll2 = train_ibm2(bitext[:50000], num_iters=10)

IBM2 Iter 1: Log-Likelihood = -805842.93
IBM2 Iter 2: Log-Likelihood = -590715.58
IBM2 Iter 3: Log-Likelihood = -521695.71
IBM2 Iter 4: Log-Likelihood = -498673.86
IBM2 Iter 5: Log-Likelihood = -490360.01
IBM2 Iter 6: Log-Likelihood = -486614.39
IBM2 Iter 7: Log-Likelihood = -484471.71
IBM2 Iter 8: Log-Likelihood = -483031.65
IBM2 Iter 9: Log-Likelihood = -482042.54
IBM2 Iter 10: Log-Likelihood = -481376.75


In [24]:
# Check the translation tables
print(t2['i']['je'])
print(t2['you']['tu'])
print(t2['you']['vous'])

0.6597546196716724
0.3836355821059363
0.492895103366037


In [25]:
t2, a2, ll2 = train_ibm2(bitext[:50000], num_iters=10, initial_t = t)

IBM2 Iter 1: Log-Likelihood = -519995.15
IBM2 Iter 2: Log-Likelihood = -505489.24
IBM2 Iter 3: Log-Likelihood = -499406.08
IBM2 Iter 4: Log-Likelihood = -495653.35
IBM2 Iter 5: Log-Likelihood = -492991.96
IBM2 Iter 6: Log-Likelihood = -491070.30
IBM2 Iter 7: Log-Likelihood = -489543.82
IBM2 Iter 8: Log-Likelihood = -488369.20
IBM2 Iter 9: Log-Likelihood = -487395.59
IBM2 Iter 10: Log-Likelihood = -486557.56


In [26]:
print(t2['i']['je'])
print(t2['you']['tu'])
print(t2['you']['vous'])

0.6648048715386161
0.39043445334165233
0.5073771070799381


### IV - Compare with the NLTK implementation

```NLTK``` has ```IBMModel1``` and ```IBMModel2``` implementations, which we can compare to our own. We can especially look at the values obtained in the translation table, and check that they correspond.
However, these models rely on ```AlignedSent``` objects for the data. Transform each pair of (source sentence $f_k$, target sentence $e_k$) into a ```AlignedSent``` object, and make the dataset a list of ```AlignedSent```:
<div class='alert alert-block alert-info'>
            Code:</div>

In [32]:
from nltk.translate import AlignedSent

# Let's try with NLTK models
def make_aligned_corpus(pairs):
    aligned_corpus = []
    for src, tgt in pairs:
        # Careful: the attribute "words" indicates "e", notation for the target
        # "mots" corresponds to "f", notation for the source
        aligned_corpus.append(AlignedSent(words=preprocess(tgt), mots=preprocess(src)))
    return aligned_corpus

In [34]:
aligned_tatoeba = make_aligned_corpus(pairs[:50000])

With that, it's very easy to train the models. Check on their attribute ```IBMModel.translation_table``` to get values of $t$. 

In [35]:
from nltk.translate.ibm1 import IBMModel1

ibm1 = IBMModel1(aligned_tatoeba, iterations=20)

In [36]:
print(ibm1.translation_table['tu']['you'])
print(ibm1.translation_table['vous']['you'])

0.38949053403099854
0.48068329388578745


In [37]:
from nltk.translate.ibm2 import IBMModel2

ibm2 = IBMModel2(aligned_tatoeba, iterations=20)

In [38]:
print(ibm2.translation_table['tu']['you'])
print(ibm2.translation_table['vous']['you'])

0.394477245785331
0.49897118247169864


### V - Implement a naïve translation function

IBM models are only a part of a functionning translation model. Notably, we need:
- A **language model**
- A decoding method

Still, we can look at the result of **word-by-word** translation: for each word $f$ in the source sentence, we pick the highest probable $e$: 
<div class='alert alert-block alert-info'>
            Code:</div>

In [27]:
# Translate sentences words by words using translation tables
# Very naive method ! 
def translate_sentence(sentence, t):
    result = []
    # For each source word
    for f in sentence:
        best_e = None
        best_prob = 0
        # Check every target word in vocabulary
        for e in t:
            # Take the best one
            if t[e][f] > best_prob:
                best_prob = t[e][f]
                best_e = e
        result.append(best_e)
    return result

In [28]:
french_sentence = ["je", "suis", "gras"]
print(translate_sentence(french_sentence, t))

['i', 'm', 'greasy']


In [29]:
french_sentence = ["je", "suis", "gras"]
print(translate_sentence(french_sentence, t2))

['i', 'm', 'greasy']


In [30]:
def translate_ibm2(f_sent, t, a):
    l_f = len(f_sent)
    output = []
    # For each source word
    for i in range(1, l_f + 1):
        best_j = None
        best_score = 0.0
        best_e = None
        # For every possible target word in vocabulary
        for j, f in enumerate(f_sent):
            for e in t:
                # Check product of translation probability and positional alignment probability
                # Use uniform probability if the positional configuration is not in the model
                score = t[e][f] * a.get((i, j, l_f, l_f), 1.0 / l_f)
                if score > best_score:
                    best_score = score
                    best_j = j
                    best_e = e
        output.append(best_e)
    return output

In [31]:
french_sentence = ["je", "suis", "très", "gras"]
print(translate_ibm2(french_sentence, t2, a2))

['i', 'm', 'greasy', 'very']


Assume that you have access to a IBM-1 model like the one we just trained, **and** to a *bigram* language model of english. Propose an **efficient** (better than naïve) procedure to find the most likely translation, making use of both of these models:

<div class='alert alert-block alert-warning'>
            Question:</div>